# PaddleOCR-VL — Document Layout Analysis report

Notebook chuẩn cho method: chạy source inference, tính Precision/Recall/F1 bằng `src/common/metrics.py`, sau đó tạo và hiển thị ảnh đối chiếu bằng `visualization/scripts/visualize_predictions.py`.

**Trạng thái hiện tại:** repository chưa có `methods/paddleocr_vl/src/run_inference.py` và runtime đang chọn chưa cài PaddleOCR/PaddlePaddle. Cell inference sẽ báo rõ tình trạng này; các cell report/visualization có thể chạy sau khi có prediction JSON đúng schema tại `outputs/predictions/paddleocr_vl/<dataset>/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "methods").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "methods").is_dir():
    raise FileNotFoundError("Không tìm thấy thư mục gốc repository; hãy mở notebook từ project DO-AN-261.")

METHOD = "paddleocr_vl"
DATASET = "doclaynet"
SPLIT = "train"
LIMIT = 20
IOU_THRESHOLD = 0.5
PREDICTIONS_DIR = ROOT / "outputs" / "predictions" / METHOD / DATASET
FIGURES_DIR = ROOT / "outputs" / "reports" / f"{METHOD}_{DATASET}"
print(f"Project: {ROOT}\nMethod: {METHOD} | Dataset: {DATASET}/{SPLIT} | Limit: {LIMIT}")

## 1. Chạy inference

Notebook gọi entrypoint của method thay vì chép lại phần xử lý model. Chưa có source thì không tạo kết quả thay thế hoặc giả lập.

In [ ]:
inference_script = ROOT / "methods" / METHOD / "src" / "run_inference.py"
if inference_script.is_file():
    command = [sys.executable, str(inference_script), "--dataset", DATASET, "--limit", str(LIMIT)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print(f"Inference chưa chạy: chưa có source {inference_script.relative_to(ROOT)}. Implement method trước; notebook không tạo prediction giả.")

## 2. Đánh giá metric

Evaluator chỉ tính trên các ảnh có prediction. Strict yêu cầu khớp cả vị trí và category; loose chỉ khớp vị trí.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from common.metrics import evaluate
from scripts.evaluate_predictions import load_gt, load_predictions
import pandas as pd
from IPython.display import display

gt_path = ROOT / "datasets" / DATASET / "processed" / "annotations" / f"{SPLIT}.json"
if not gt_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy ground truth: {gt_path}")
pred_by_image = load_predictions(PREDICTIONS_DIR)
if not pred_by_image:
    print(f"Chưa có predictions trong {PREDICTIONS_DIR}. Hoàn tất inference hoặc đặt prediction JSON đúng schema rồi chạy lại cell này.")
else:
    gt_by_image, category_names = load_gt(gt_path)
    gt_subset = {image_id: boxes for image_id, boxes in gt_by_image.items() if image_id in pred_by_image}
    strict = evaluate(gt_subset, pred_by_image, category_names, IOU_THRESHOLD, require_same_category=True)
    loose = evaluate(gt_subset, pred_by_image, category_names, IOU_THRESHOLD, require_same_category=False)
    print(f"Đánh giá {len(gt_subset)} ảnh; IoU >= {IOU_THRESHOLD}")
    def overall_row(name, result):
        values = result["overall"]
        return {"Metric": name, "Precision": values["precision"], "Recall": values["recall"], "F1": values["f1"], "TP": values["tp"], "FP": values["fp"], "FN": values["fn"], "Mean IoU (matched)": values["mean_iou_matched"]}
    display(pd.DataFrame([overall_row("Strict (vị trí + nhãn)", strict), overall_row("Loose (chỉ vị trí)", loose)]).set_index("Metric").round(3))
    rows = [{"ID": category_id, "Category": values["name"], "Support": values["support"], "TP": values["tp"], "FP": values["fp"], "FN": values["fn"], "Precision": values["precision"], "Recall": values["recall"], "F1": values["f1"]} for category_id, values in strict["per_category"].items()]
    display(pd.DataFrame(rows).sort_values("Support", ascending=False).set_index("Category").round(3))

## 3. Ground Truth vs. prediction

Dùng visualizer hiện có của project; cần có prediction JSON và ground truth tương ứng.

In [ ]:
from IPython.display import Image as DisplayImage, display

prediction_files = list(PREDICTIONS_DIR.glob("*.json"))
if not prediction_files:
    print("Chưa có prediction để trực quan hóa. Cell này sẽ chạy được sau khi inference tạo JSON.")
else:
    visualizer = ROOT / "visualization" / "scripts" / "visualize_predictions.py"
    command = [sys.executable, str(visualizer), "--method", METHOD, "--dataset", DATASET, "--split", SPLIT, "--limit", str(LIMIT), "--output-dir", str(FIGURES_DIR)]
    subprocess.run(command, cwd=ROOT, check=True)
    figure_files = sorted(FIGURES_DIR.glob("*.png"))
    if figure_files:
        print(figure_files[0])
        display(DisplayImage(filename=str(figure_files[0])))
    else:
        print(f"Visualizer không tạo ảnh nào trong {FIGURES_DIR}")

## Files đầu ra

- Predictions: `outputs/predictions/paddleocr_vl/doclaynet/`
- Hình đối chiếu: `outputs/reports/paddleocr_vl_doclaynet/`
- Ground Truth: `datasets/doclaynet/processed/annotations/train.json`

Định dạng prediction mà evaluator cần: mỗi file JSON có `image_id` và `detections`; mỗi detection gồm `category_id` và `bbox: [x, y, width, height]`.